# Super Notebook pour prédire des enzymes

### Imports & parameters

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor  
from sklearn.metrics import r2_score
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import Ridge
from scipy.stats import pearsonr
from sklearn.metrics import r2_score, roc_auc_score, f1_score, precision_recall_curve



# -----------------------------------------------------
# --- Variables ---------------------------------------
# -----------------------------------------------------

SEUIL_NOCETO_MAX = 1000
SEUIL_BRADFORD_MIN = 0.1
SEUIL_SUPPRESSION = 0.90 
UTILISER_LOG_SUR_Y = False 
ENZYME_RECHERCHEE = "Q5N2G0"

# Create a true 

# Autres variables non inclus ici:
# Le nombre de variables du ESM: 10 -> 91.29% (480 -> 100%)
# A corriger: présente une forme de fuite non-supervisée lors ce que la PCA réduit les dimensions.
# Il faut donc faire la PCA sur le train et appliquer la transformation sur le test (négligeable pour l'instant).

### Data Preprocessing

In [2]:
# -----------------------------------------------------
# --- Cleaning ----------------------------------------
# -----------------------------------------------------

df = pd.read_csv("data_exploitable_python_05 (merged).csv", sep=";", decimal=".")

ESM2 = []

Zscales = [
    "Col252_Z1_Pol", "Col252_Z2_Vol", "Col252_Z3_Elec", "Col313_Z1_Pol", "Col313_Z2_Vol",
    "Col313_Z3_Elec", "Col336_Z1_Pol", "Col336_Z2_Vol", "Col336_Z3_Elec", "Col371_Z1_Pol",
    "Col371_Z2_Vol", "Col371_Z3_Elec", "Col467_Z1_Pol", "Col467_Z2_Vol", "Col467_Z3_Elec",
    "Col502_Z1_Pol", "Col502_Z2_Vol", "Col502_Z3_Elec", "Col553_Z1_Pol", "Col553_Z2_Vol",
    "Col553_Z3_Elec", "Col623_Z1_Pol", "Col623_Z2_Vol", "Col623_Z3_Elec", "Col624_Z1_Pol",
    "Col624_Z2_Vol", "Col624_Z3_Elec"
]

#  On place le Matricule en index pour ne pas le perdre, mais le cacher à l'algorithme
df.set_index("Acc. Number", inplace=True)
df_cible = df[df.index == ENZYME_RECHERCHEE].copy()
df = df[df.index != ENZYME_RECHERCHEE].copy()

df['Activite_Brute'] = pd.to_numeric(df['Activite_Brute'], errors='coerce')
df['NoCeto'] = pd.to_numeric(df['NoCeto'], errors='coerce')


# Filtrage sur les Gels (exclusion stricte des enzymes non induites ET non exprimées)
# df = df[~((df['Induction (Gel)'] == 0) & (df['Expression (Gel)'] == 0))].copy()

# Supprimer les enzymes avec un bruit de fond non spécifique trop élevé (>= 0.04)
df = df[df['NoCeto'] < SEUIL_NOCETO_MAX].copy()
df = df[df['Bradford (mg/mL)'] > SEUIL_BRADFORD_MIN].copy()

groups = df.index.to_series()

# Traitement biologique

df['Activite_Brute'] = df['Activite_Brute'] - df['NoCeto']             # Correction du blanc
df.loc[df['Activite_Brute'] <= 0.05, 'Activite_Brute'] = 0.0           # Filtre du bruit analytique
df['Activite_Brute'] = df['Activite_Brute'] / df['Bradford (mg/mL)']   # Normalisation en Activité Spécifique

# Nettoyage final des colonnes inutiles
colonnes_a_supprimer = ['NoCeto', 'Bradford (mg/mL)', 'Induction (Gel)', 'Expression (Gel)', 'Annotation', 'TAM Ref.', 'Sequence', 'Substrat']

X = df.drop(columns=colonnes_a_supprimer + ['Activite_Brute'], errors='ignore')  # Variables d'entrée (features propres)
y = np.log1p(df['Activite_Brute']) if UTILISER_LOG_SUR_Y else df['Activite_Brute']  # Variable de sortie (target)

print(f"Données prêtes : {X.shape[0]} interactions, {X.shape[1]} variables X.")


# -----------------------------------------------------
# --- Splitting ---------------------------------------
# -----------------------------------------------------

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# On récupère les indices des lignes pour le Train et le Test
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

print(f"Lignes d'Entraînement : {len(X_train)} (soit {groups_train.nunique()} enzymes uniques)")
print(f"Lignes de Test        : {len(X_test)} (soit {groups_test.nunique()} enzymes CACHÉES)")

# Additional post-splitting cleaning: Suppression des colonnes Z-Scales avec une valeur dominante (fréquence >= SEUIL_SUPPRESSION)
colonnes_a_supprimer = []

for col in Zscales:
        if col in df.columns:
            # Calcule la fréquence de la valeur la plus commune (inclut les NaN dans le calcul)
            max_freq = X_train[col].value_counts(normalize=True, dropna=False).max()
            
            # Si la fréquence dépasse le seuil, on marque la colonne pour suppression
            if max_freq >= SEUIL_SUPPRESSION:
                print(f"Colonne {col} : fréquence max = {max_freq:.2f}")
                colonnes_a_supprimer.append(col)

print(f"Filtre de conservation : {len(colonnes_a_supprimer)} colonnes Z-scales supprimées sur {len(Zscales)}.")

X_train = X_train.drop(columns=colonnes_a_supprimer)
X_test = X_test.drop(columns=colonnes_a_supprimer)

modeles = {
    "Random Forest": RandomForestRegressor(
        n_estimators=500, 
        max_depth=12,          
        min_samples_leaf=5,   
        max_features=0.6, 
        n_jobs=-1, 
        random_state=42
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=500, 
        max_depth=20, 
        min_samples_leaf=4, 
        max_features=0.6, 
        n_jobs=-1, 
        random_state=42
    ),
    "Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=300, 
        learning_rate=0.03, 
        max_depth=8,
        max_leaf_nodes=25, 
        min_samples_leaf=15, 
        l2_regularization=1.0,  # Freine le surapprentissage sur les poids élevés
        random_state=42
    )
}

substrats_du_train = df["Substrat"].iloc[train_idx].values

poids_train = np.ones(len(y_train))
for sub in np.unique(substrats_du_train):
    masque_s = (substrats_du_train == sub)
    std_s = np.std(y_train.values[masque_s])
    if std_s > 1e-6:
        poids_train[masque_s] = 1.0 / (std_s ** 2)
poids_train = np.clip(poids_train / np.mean(poids_train), 0.1, 10.0) # Sécurité anti-poids extrêmes

Données prêtes : 7497 interactions, 55 variables X.
Lignes d'Entraînement : 5984 (soit 352 enzymes uniques)
Lignes de Test        : 1513 (soit 89 enzymes CACHÉES)
Filtre de conservation : 0 colonnes Z-scales supprimées sur 27.


### Model training

In [3]:
# ---------------------------------------------------------
# --- ENTRAÎNEMENT ET ÉVALUATION --------------------------
# ---------------------------------------------------------

y_pred = {}
pipelines = {}
correctifs_biais = {}
substrats_du_train = df["Substrat"].iloc[train_idx].values
substrats_du_test = df["Substrat"].iloc[test_idx].values
liste_substrats_uniques = np.unique(substrats_du_test)

for ai_name, algo in modeles.items():
    pipeline  = Pipeline([
        ("scaler", StandardScaler()),                                     
        ("algo", algo)
    ])

    print(f"\nEntraînement de l'IA {ai_name} en cours...")
    pipeline.fit(X_train, y_train, algo__sample_weight=poids_train)

    pipelines[ai_name] = pipeline

    pred_test_brute = pipeline.predict(X_test)

    print(f"Étalonnage du correctif de biais pour {ai_name}...")
    pred_test_corrigee = np.zeros_like(pred_test_brute)
    
    SEUIL_BRUIT_IA = 0.025 

    for sub in np.unique(substrats_du_train):
        m_tr = (substrats_du_train == sub)
        m_te = (substrats_du_test == sub)
        
        if not np.any(m_te):
            continue
            
        preds_sub = pred_test_brute[m_te].copy()
        
        preds_sub = np.where(preds_sub < SEUIL_BRUIT_IA, 0.0, preds_sub)
        
        vrai_max = np.max(y_train.values[m_tr])
        pred_max = np.max(preds_sub)
        
        if vrai_max > 0.01 and pred_max > 0.01:
            facteur_etirement = vrai_max / pred_max
            facteur_etirement = min(facteur_etirement, 5.0) 
            
            preds_sub = preds_sub * facteur_etirement
            
        pred_test_corrigee[m_te] = preds_sub
        
    y_pred[f"{ai_name}"] = pred_test_brute
    y_pred[f"{ai_name} (Corrigé)"] = pred_test_corrigee


Entraînement de l'IA Random Forest en cours...
Étalonnage du correctif de biais pour Random Forest...

Entraînement de l'IA Extra Trees en cours...
Étalonnage du correctif de biais pour Extra Trees...

Entraînement de l'IA Gradient Boosting en cours...
Étalonnage du correctif de biais pour Gradient Boosting...


### Assessment

In [4]:
result_liste = [
    "Random Forest",
    "Extra Trees",
    "Gradient Boosting"
]

SEUIL_ACTIF = np.log1p(0.05) if UTILISER_LOG_SUR_Y else 0.05

for ai_name in result_liste:
    predictions = y_pred[ai_name]

    print(f"\n" + "="*70)
    print(f"🧬 RÉSULTATS DÉTAILLÉS : {ai_name.upper()}")
    print("="*70)
    print(f"{'SUBSTRAT':10s} | {'R²':>7s} | {'AUC':>7s} | {'F1-Score':>8s} | {'Nb Actifs / Total'}")
    print("-" * 70)

    scores_r2, scores_auc, scores_f1 = {}, {}, {}

    for sub in liste_substrats_uniques:
        masque = (substrats_du_test == sub)
        y_test_sub = y_test.values[masque]
        y_pred_sub = predictions[masque]
        
        if len(y_test_sub) < 2:
            continue
            
        r2 = r2_score(y_test_sub, y_pred_sub)
        if r2 > 0:
            scores_r2[sub] = r2
            
        y_test_bin = (y_test_sub >= SEUIL_ACTIF).astype(int)
        nb_actifs = sum(y_test_bin)
        nb_total = len(y_test_bin)
        
        if 0 < nb_actifs < nb_total:
            auc = roc_auc_score(y_test_bin, y_pred_sub)
            scores_auc[sub] = auc
            
            # Calcul du F1-score optimal pour ce modèle
            prec, rap, _ = precision_recall_curve(y_test_bin, y_pred_sub)
            f1_max = np.max(2 * (prec * rap) / (prec + rap + 1e-10))
            scores_f1[sub] = f1_max
            
            print(f"{sub:10s} | {r2:7.3f} | {auc:7.3f} | {f1_max:8.3f} | {nb_actifs:3d} / {nb_total:3d}")
        else:
            print(f"{sub:10s} | {r2:7.3f} | {'N/A':>7s} | {'N/A':>8s} | {nb_actifs:3d} / {nb_total:3d}")

    print(f"\n--- BILANS MOYENS ({ai_name}) ---")
    print(f"🎯 R² moyen (viables) : {np.mean(list(scores_r2.values())):.3f} ({len(scores_r2)}/17 substrats)")
    print(f"🔎 AUC moyenne        : {np.mean(list(scores_auc.values())):.3f}")
    print(f"⚖️ F1-Score moyen     : {np.mean(list(scores_f1.values())):.3f}")


🧬 RÉSULTATS DÉTAILLÉS : RANDOM FOREST
SUBSTRAT   |      R² |     AUC | F1-Score | Nb Actifs / Total
----------------------------------------------------------------------
2KDGA      |   0.000 |     N/A |      N/A |   0 /  89
2O2PAA     |   0.012 |   0.886 |    0.444 |   6 /  89
2O4PBA     |  -0.062 |   0.736 |    0.545 |   5 /  89
2OATA      |   0.103 |   0.777 |    0.545 |  14 /  89
2OHAA      |   0.020 |   0.715 |    0.509 |  21 /  89
2OPAA      |   0.014 |   0.747 |    0.485 |  14 /  89
3.3diMe    |  -0.187 |     N/A |      N/A |   0 /  89
3H2OBA     |  -0.073 |     N/A |      N/A |   0 /  89
4H2OA      |   0.028 |   0.814 |    0.500 |   3 /  89
4H4PA      |   0.064 |   0.663 |    0.415 |  17 /  89
4MT2OB     |   0.022 |   0.577 |    0.726 |  49 /  89
BOHPh      |   0.019 |   0.902 |    0.400 |   2 /  89
BOHiPr     |   0.005 |   0.900 |    0.400 |   4 /  89
HPA        |   0.027 |   0.891 |    0.333 |   3 /  89
MOPA       |   0.030 |   0.728 |    0.410 |  11 /  89
Pyr        |  -0.0

In [5]:
# On prépare les colonnes X de Q5N2G0 exactement comme X_test
X_cible = df_cible[X_train.columns]

pred_log = pipelines["Extra Trees"].predict(X_cible)
pred_activite = np.expm1(pred_log) if UTILISER_LOG_SUR_Y else pred_log

df_resultats_cible = pd.DataFrame({
    "Substrat": df_cible["Substrat"].values,
    "Activite_Specifique_Predite": np.round(pred_activite, 3)
}).sort_values(by="Activite_Specifique_Predite", ascending=False)

print(f"Profil d'activité prédit pour {ENZYME_RECHERCHEE} :")
print(df_resultats_cible.to_string(index=False))

Profil d'activité prédit pour Q5N2G0 :
Substrat  Activite_Specifique_Predite
   2OATA                        0.128
   4H4PA                        0.053
  4MT2OB                        0.050
     Pyr                        0.030
     HPA                        0.023
   2OHAA                        0.014
   4H2OA                        0.006
  2O4PBA                        0.004
   2OPAA                        0.002
  2O2PAA                        0.002
    MOPA                        0.001
  tMePyr                        0.000
  3H2OBA                        0.000
   BOHPh                        0.000
  BOHiPr                        0.000
   2KDGA                        0.000
 3.3diMe                        0.000


### Export data

In [6]:
# from datetime import datetime

# # Génère une chaîne de texte au format "2026-10-20_10h42"
# timestamp = datetime.now().strftime("%Y-%m-%d_%Hh%M")

# # Sauvegarde le fichier avec le nom horodaté
# nom_fichier = f"dataset_propre_{timestamp}.csv"
# df.to_csv(nom_fichier, sep=";", decimal=",", index=False, encoding="utf-8")

# print(f"Dataset exporté avec succès sous : {nom_fichier}")